In [0]:
create or replace materialized view fragrance_db.default.mv_cleaned_frag  as (
 WITH ranked AS (
        SELECT
            REGEXP_EXTRACT(url, '([a-zA-Z0-9]+)\\.html$', 1) as id,
            replace(regexp_extract(url, '/perfume/[^/]+/([^/]+)-[0-9]+\\.html', 1), '-', ' ') as name,
            replace(regexp_extract(url, '/perfume/([^/]+)/', 1), '-', ' ') as brand,
            REGEXP_EXTRACT(description, 'was launched in ([0-9]{4})', 1) as release_year,
            replace(replace(replace(perfumers, '[', ''), ']', ''),"'", "") as perfumers,
            CASE
                WHEN name LIKE '%for women and men' THEN 'unisex'
                WHEN name LIKE '%for women' THEN 'women'
                WHEN name LIKE '%for men' THEN 'men'
                ELSE NULL
            END as gender,
            TRY_CAST(REGEXP_REPLACE(rating_count, ',', '') AS INT) as rating_count,
            TRY_CAST(rating AS DECIMAL(10,2)) as rating,
            replace(replace(replace(main_accords, '[', ''), ']', ''),"'", "") as accords,
            lower(replace(regexp_extract(description, '(?i)top note[s]? (is|are) ([^.;]*)', 2), ' and', ',')) as top_notes,
            lower(replace(regexp_extract(description, '(?i)middle note[s]? (is|are) ([^.;]*)', 2), ' and', ',')) as mid_notes,
            lower(replace(regexp_extract(description, '(?i)base note[s]? (is|are) ([^.;]*)', 2), ' and', ',')) as base_notes,
            description,
            url,
            row_number() over (partition by url order by rating_count desc) as rn
        FROM fragrance_db.default.frag_raw
    )
    SELECT 
        *,
        TRIM(
        CONCAT(
            CASE WHEN accords IS NOT NULL AND accords != ''
                THEN ' ' || array_join(
                    transform(split(accords, ','), x -> concat('accords_', replace(trim(x), ' ', '_'))),
                    ' '
                )
                ELSE ''
            END,
            CASE WHEN top_notes IS NOT NULL AND top_notes != ''
                THEN ' ' || array_join(
                    transform(split(top_notes, ','), x -> concat('top_notes_', replace(trim(x), ' ', '_'))),
                    ' '
                )
                ELSE ''
            END,
            CASE WHEN mid_notes IS NOT NULL AND mid_notes != ''
                THEN ' ' || array_join(
                    transform(split(mid_notes, ','), x -> concat('mid_notes_', replace(trim(x), ' ', '_'))),
                    ' '
                )
                ELSE ''
            END,
            CASE WHEN base_notes IS NOT NULL AND base_notes != ''
                THEN ' ' || array_join(
                    transform(split(base_notes, ','), x -> concat('base_notes_', replace(trim(x), ' ', '_'))),
                    ' '
                )
                ELSE ''
            END
        )
    ) AS perfume_string
    FROM ranked
    WHERE rn = 1
)
;